In [2]:
import pandas as pd
import numpy as np

# df = pd.read_csv("datasets\Diseases_and_Symptoms_dataset.csv")
df = pd.read_csv("datasets\Final_Augmented_dataset_Diseases_and_Symptoms.csv")

print(df.shape)
print(df.head())

(246945, 378)
         diseases  anxiety and nervousness  depression  shortness of breath  \
0  panic disorder                        1           0                    1   
1  panic disorder                        0           0                    1   
2  panic disorder                        1           1                    1   
3  panic disorder                        1           0                    0   
4  panic disorder                        1           1                    0   

   depressive or psychotic symptoms  sharp chest pain  dizziness  insomnia  \
0                                 1                 0          0         0   
1                                 1                 0          1         1   
2                                 1                 0          1         1   
3                                 1                 0          1         1   
4                                 0                 0          0         1   

   abnormal involuntary movements  chest t

In [3]:
X = df.drop("diseases", axis=1)
y = df["diseases"]

In [4]:
print(df.shape)
print(df.columns)
print(df['diseases'].nunique())

(246945, 378)
Index(['diseases', 'anxiety and nervousness', 'depression',
       'shortness of breath', 'depressive or psychotic symptoms',
       'sharp chest pain', 'dizziness', 'insomnia',
       'abnormal involuntary movements', 'chest tightness',
       ...
       'stuttering or stammering', 'problems with orgasm', 'nose deformity',
       'lump over jaw', 'sore in nose', 'hip weakness', 'back swelling',
       'ankle stiffness or tightness', 'ankle weakness', 'neck weakness'],
      dtype='object', length=378)
773


In [5]:
from sklearn.model_selection import train_test_split

# Keep only classes with at least 2 samples
valid_classes = y[y.isin(y.value_counts()[y.value_counts() >= 2].index)]
X_filtered = X.loc[valid_classes.index]
y_filtered = y.loc[valid_classes.index]

X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_filtered,
    test_size=0.2,
    random_state=42,
    stratify=y_filtered
)

In [6]:
print("Classes in y_test:", len(np.unique(y_test)))
print("Classes in y_train:", len(np.unique(y_train)))
print("Classes in X_train:", len(np.unique(X_train)))
print("Classes in X_test:", len(np.unique(X_test)))


Classes in y_test: 740
Classes in y_train: 754
Classes in X_train: 2
Classes in X_test: 2


In [7]:
from sklearn.naive_bayes import BernoulliNB

nb = BernoulliNB()
nb.fit(X_train, y_train)

pred = nb.predict(X_test)

In [8]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

In [9]:
from sklearn.metrics import accuracy_score, f1_score

print("NB Accuracy:", accuracy_score(y_test, pred))
print("LR Accuracy:", accuracy_score(y_test, pred_lr))

print("LR Macro F1:", f1_score(y_test, pred_lr, average="macro"))

NB Accuracy: 0.8496739966792208
LR Accuracy: 0.8670878386587292
LR Macro F1: 0.7977671155268197


In [10]:
from sklearn.metrics import top_k_accuracy_score

# Ensure y_test only includes classes seen in training
y_test_nb = y_test[y_test.isin(nb.classes_)]
X_test_nb = X_test.loc[y_test_nb.index]

y_test_lr = y_test[y_test.isin(lr.classes_)]
X_test_lr = X_test.loc[y_test_lr.index]

nb_probs = nb.predict_proba(X_test_nb)
lr_probs = lr.predict_proba(X_test_lr)

In [11]:
print("NB Top-3:", top_k_accuracy_score(y_test_nb, nb_probs, k=3, labels=nb.classes_))
print("LR Top-3:", top_k_accuracy_score(y_test_lr, lr_probs, k=3, labels=lr.classes_))

NB Top-3: 0.9456931114080913
LR Top-3: 0.9627019803183088


In [12]:
print(y.value_counts())

diseases
cystitis                          1219
nose disorder                     1218
vulvodynia                        1218
complex regional pain syndrome    1217
spondylosis                       1216
                                  ... 
open wound of the head               1
myocarditis                          1
chronic ulcer                        1
hypergammaglobulinemia               1
kaposi sarcoma                       1
Name: count, Length: 773, dtype: int64


In [13]:
from sklearn.svm import LinearSVC

svm = LinearSVC()

svm.fit(X_train, y_train)

pred_svm = svm.predict(X_test)

from sklearn.metrics import accuracy_score

print("SVM Accuracy:", accuracy_score(y_test, pred_svm))

SVM Accuracy: 0.8647794921637711


In [14]:
X_test_missing = X_test.copy()
X_test_missing.iloc[:, :] = X_test_missing.iloc[:, :] * (np.random.rand(*X_test.shape) > 0.2)

In [15]:
df.duplicated().sum()

np.int64(57298)

In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score
import numpy as np

# Convert binary features to uint8 to reduce memory pressure
X_train = X_train.astype(np.uint8)
X_test = X_test.astype(np.uint8)

# Use a small random subset for training if memory is limited
subset_fraction = 0.05  # reduce training size to 5%
subset_size = max(1000, int(len(X_train) * subset_fraction))
idx = np.random.RandomState(42).choice(X_train.index, size=subset_size, replace=False)
X_train_small = X_train.loc[idx].astype(np.float32)
y_train_small = y_train.loc[idx]
X_test_small = X_test.astype(np.float32)

rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=15,
    class_weight='balanced',
    n_jobs=1,
    random_state=42
)
rf.fit(X_train_small, y_train_small)

y_pred = rf.predict(X_test_small)
probs = rf.predict_proba(X_test_small)

print("RF Accuracy:", accuracy_score(y_test, y_pred))
print("RF Macro F1:", f1_score(y_test, y_pred, average="macro"))

# Evaluate top-k only on test samples with labels seen by the model
y_test_rf = y_test[y_test.isin(rf.classes_)]
X_test_rf = X_test_small.loc[y_test_rf.index]
probs_rf = rf.predict_proba(X_test_rf)

print("RF Top-3:", top_k_accuracy_score(y_test_rf, probs_rf, k=3, labels=rf.classes_))

RF Accuracy: 0.5716194873040943
RF Macro F1: 0.4483612669858036
RF Top-3: 0.7179890826569624


In [22]:
from sklearn.linear_model import SGDClassifier
from sklearn.multiclass import OneVsRestClassifier

# Use a smaller subset for memory efficiency
subset_fraction = 0.1  # 10% of training data
subset_size = max(2000, int(len(X_train) * subset_fraction))
idx = np.random.RandomState(42).choice(X_train.index, size=subset_size, replace=False)
X_train_small = X_train.loc[idx]
y_train_small = y_train.loc[idx]

svm = OneVsRestClassifier(
    SGDClassifier(loss="log_loss", class_weight="balanced", max_iter=1000, tol=1e-3),
    n_jobs=1  # Reduce to 1 for memory
)
svm.fit(X_train_small, y_train_small)

y_pred_svm = svm.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))

# For top-k, filter to seen classes
y_test_svm = y_test[y_test.isin(svm.classes_)]
X_test_svm = X_test.loc[y_test_svm.index]
probs_svm = svm.predict_proba(X_test_svm)

print("SVM Top-3:", top_k_accuracy_score(y_test_svm, probs_svm, k=3, labels=svm.classes_))

C:\Users\krshn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
C:\Users\krshn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
C:\Users\krshn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
C:\User

SVM Accuracy: 4.0497306929089216e-05
SVM Top-3: 0.27689558542825815


In [1]:
import pandas as pd
import numpy as np

# Read with uint8 dtype for all binary columns — 8x less memory
df = pd.read_csv(
    "datasets\Final_Augmented_dataset_Diseases_and_Symptoms.csv",
    dtype={col: 'uint8' for col in pd.read_csv(
        "datasets\Final_Augmented_dataset_Diseases_and_Symptoms.csv", nrows=0
    ).columns if col != 'diseases'}
)

print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
# Should drop from ~745MB → ~93MB

Memory usage: 111.5 MB


In [6]:
import pandas as pd
import numpy as np
import re
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score
from sklearn.preprocessing import LabelEncoder

# ── 1. Load ──────────────────────────────────────────────────────────────────
df = pd.read_csv(r"datasets\Final_Augmented_dataset_Diseases_and_Symptoms.csv")

# ── 2. Remove rare classes ────────────────────────────────────────────────────
class_counts = df['diseases'].value_counts()
valid_classes = class_counts[class_counts >= 2].index
df = df[df['diseases'].isin(valid_classes)]
print(f"Dataset shape after filtering: {df.shape}")
print(f"Unique diseases: {df['diseases'].nunique()}")

# ── 3. Cast binary columns to uint8 ──────────────────────────────────────────
feature_cols = [c for c in df.columns if c != 'diseases']
df[feature_cols] = df[feature_cols].astype('uint8')

# ── 4. Clean column names (fixes LightGBM JSON character error) ───────────────
df.columns = [re.sub(r'[^A-Za-z0-9_\s]', '', col).strip() for col in df.columns]
df.columns = [col.replace(' ', '_') for col in df.columns]

# Redefine feature_cols after renaming
feature_cols = [c for c in df.columns if c != 'diseases']
print(f"\nSample cleaned column names: {feature_cols[:5]}")

# ── 5. Split features and labels ──────────────────────────────────────────────
X = df[feature_cols]
le = LabelEncoder()
y = le.fit_transform(df['diseases'])

# ── 6. Train/test split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(f"\nTrain size: {X_train.shape}, Test size: {X_test.shape}")

# ── 7. Train LightGBM ─────────────────────────────────────────────────────────
model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    n_jobs=-1,
    random_state=42,
    verbose=-1
)

model.fit(X_train, y_train)

Dataset shape after filtering: (246926, 378)
Unique diseases: 754

Sample cleaned column names: ['anxiety_and_nervousness', 'depression', 'shortness_of_breath', 'depressive_or_psychotic_symptoms', 'sharp_chest_pain']

Train size: (197540, 377), Test size: (49386, 377)


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [7]:
import joblib

# Save
joblib.dump(model, 'models/lgbm_model.pkl')
joblib.dump(le, 'models/label_encoder.pkl')  # save this too — you need it to decode predictions

print("Model saved.")

Model saved.


In [ ]:
# ── 8. Evaluate ───────────────────────────────────────────────────────────────
pred = model.predict(X_test)
probs = model.predict_proba(X_test)

print("\n── Results ──────────────────────────────────────")
print(f"Top-1 Accuracy : {accuracy_score(y_test, pred):.4f}")
print(f"Top-3 Accuracy : {top_k_accuracy_score(y_test, probs, k=3, labels=np.arange(len(le.classes_))):.4f}")
print(f"Macro F1       : {f1_score(y_test, pred, average='macro'):.4f}")

# ── 9. Robustness test (missing symptoms) ─────────────────────────────────────
print("\n── Robustness (missing symptoms) ────────────────")
for noise in [0.1, 0.2, 0.3, 0.5]:
    X_noisy = X_test.copy().values * (np.random.rand(*X_test.shape) > noise)
    noisy_pred = model.predict(X_noisy)
    noisy_probs = model.predict_proba(X_noisy)
    print(f"Top-1: {accuracy_score(y_test, noisy_pred):.4f}"
      f"Top-3: {top_k_accuracy_score(y_test, noisy_probs, k=3, labels=np.arange(len(le.classes_))):.4f}")

# ── 10. Per-disease breakdown (worst performers) ──────────────────────────────
from sklearn.metrics import classification_report

report_df = pd.DataFrame(
    classification_report(y_test, pred, target_names=le.classes_, output_dict=True)
).T

print("\n── 10 Worst Performing Diseases (by F1) ─────────")
print(report_df.drop(['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
               .sort_values('f1-score')
               .head(10)[['precision', 'recall', 'f1-score', 'support']]
               .to_string())

# ── 11. Feature importance (top 20 symptoms) ──────────────────────────────────
importances = pd.Series(
    model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n── Top 20 Most Predictive Symptoms ──────────────")
print(importances.head(20).to_string())


── Results ──────────────────────────────────────
Top-1 Accuracy : 0.1167
Top-3 Accuracy : 0.1237
Macro F1       : 0.0779

── Robustness (missing symptoms) ────────────────


C:\Users\krshn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\krshn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
